# CloudCompute: MobileNet / ViT robust fine-tuning
Отдельная ветка моделей на усиленных аугментациях. Для смены архитектуры измените только `MODEL`.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
MODEL = "vit"  # vit | mobilenet
QUICK_RUN = True
AUGMENTATION_PROFILE = "robust"  # standard | robust
EPOCHS = 3
LEARNING_RATE = None  # None = architecture default
TRAIN_BATCH_SIZE = None  # ViT default: 16; MobileNet: 64
VALIDATION_BATCH_SIZE = None  # ViT default: 32; MobileNet: 128
NUM_WORKERS = 2
RESUME_TRAINING = True
PROMOTE_ROBUST_CHAMPION = not QUICK_RUN and AUGMENTATION_PROFILE == "robust"
RUN_TESTS = False
REPO_DIR = "/root/text-orientation-classification"
STATE_DIR = "/root/text-orientation-state"

In [ ]:
import json, os, platform, shutil, subprocess, sys, torch
from datetime import datetime, timezone
from pathlib import Path

repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
MODEL_OPTIONS = {
    "mobilenet": {"name": "mobilenet_v3_large", "config": "configs/baseline.yaml", "lr": 2e-5, "train_batch": 64, "validation_batch": 128},
    "vit": {"name": "vit_b_16", "config": "configs/vit_b_16.yaml", "lr": 1e-5, "train_batch": 16, "validation_batch": 32},
}
if MODEL not in MODEL_OPTIONS:
    raise ValueError("MODEL must be 'vit' or 'mobilenet'")
if AUGMENTATION_PROFILE not in {"standard", "robust"}:
    raise ValueError("AUGMENTATION_PROFILE must be 'standard' or 'robust'")
model_option = MODEL_OPTIONS[MODEL]
model_name, config_path = model_option["name"], model_option["config"]
learning_rate = LEARNING_RATE or model_option["lr"]
train_batch_size = TRAIN_BATCH_SIZE or model_option["train_batch"]
validation_batch_size = VALIDATION_BATCH_SIZE or model_option["validation_batch"]
from src.registry import select_champion
bundle = select_champion(Path(STATE_DIR) / "registry", model_name)
print({"gpu": torch.cuda.get_device_name(0), "model": model_name, "config": config_path, "initial_checkpoint": str(bundle.checkpoint_path), "train_batch_size": train_batch_size})

In [ ]:
mode = "quick" if QUICK_RUN else "full"
run_name = f"{model_name}_{AUGMENTATION_PROFILE}_{mode}"
run_dir = Path("artifacts/experiments") / run_name
recovery_dir = Path(STATE_DIR) / "training/recovery" / AUGMENTATION_PROFILE / model_name / mode
command = [
    sys.executable, "-m", "scripts.train_robust",
    "--config", config_path,
    "--output-dir", str(run_dir),
    "--recovery-dir", str(recovery_dir),
    "--augmentation-profile", AUGMENTATION_PROFILE,
    "--initial-checkpoint", str(bundle.checkpoint_path),
    "--epochs", str(2 if QUICK_RUN else EPOCHS),
    "--learning-rate", str(learning_rate),
    "--batch-size", str(train_batch_size),
    "--validation-batch-size", str(validation_batch_size),
    "--num-workers", str(NUM_WORKERS),
]
if QUICK_RUN:
    command += ["--train-base-samples", "2048", "--validation-base-samples", "512"]
if RESUME_TRAINING:
    command.append("--resume")
subprocess.run(command, check=True)

In [ ]:
environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}
(run_dir / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
if not QUICK_RUN:
    subprocess.run([sys.executable, "-m", "scripts.calibrate", "--run-dir", str(run_dir), "--config", config_path], check=True)
    if PROMOTE_ROBUST_CHAMPION:
        subprocess.run([sys.executable, "-m", "scripts.promote_robust_champion", "--run-dir", str(run_dir), "--project-dir", STATE_DIR], check=True)
        subprocess.run([sys.executable, "-m", "scripts.compare_robust", "--project-dir", STATE_DIR, "--model", model_name, "--minimum-improvement", "0.005"], check=True)
exports = Path(STATE_DIR) / "training/runs" / AUGMENTATION_PROFILE / model_name / mode
exports.mkdir(parents=True, exist_ok=True)
archive = Path(shutil.make_archive(str(exports / run_name), "zip", root_dir=run_dir))
print("Result:", archive)
print("Standard registry unchanged.")